# A narrow HR scope

Use this numbered folder and the shared root kernel. The assistant covers Cedar Health leave and benefits. Inputs and separate evaluation expectations are AI-authored drafts, not human judgments.

In [ ]:
import json
from pathlib import Path
from pprint import pprint
fixtures = json.loads(Path("docs/dataset/inputs.json").read_text())
expectations = json.loads(Path("docs/goldends/draft_expectations.json").read_text())
pprint(fixtures)

## Read the scope helper before importing

`tools/topic.py` checks whole words, returns a decision/reason, and calls a responder only after allow. `models/demo.py` echoes accepted requests and is deliberately labelled local demo code. No classifier runs.

In [ ]:
from tools.topic import check_topic, request
from models.demo import demo_response
pprint(check_topic(fixtures[0]["text"]))

## Run the normal, problematic, and edge inputs

Inspect the final exposed text and actual local input log. Blocked requests have zero local invocations; allowed ones have one. These counts are not API calls.

In [ ]:
observed = []
baseline = {item["id"]: request(item["text"], demo_response, observed) for item in fixtures}
for name, result in baseline.items():
    pprint({"id": name, **result})
pprint({"actual_local_invocations": len(observed), "inputs_seen": observed, "model_api_calls": 0})

## Inspect usefulness and misses

The legitimate vacation wording is accidentally blocked. The unrelated pasta and poker questions pass because they contain scope words. Their exposed echoes are actual local function outputs, not model behavior.

In [ ]:
pprint({name: baseline[name] for name in ["borderline", "leave_collision", "benefits_collision"]})
assert baseline["borderline"]["decision"] == "block"
assert baseline["leave_collision"]["decision"] == "allow"

## Change one condition and replay the same inputs

Enable only `include_vacation`; keep all fixtures, the original keywords, and responder unchanged. This improves the borderline example but cannot distinguish meanings of leave or benefits.

In [ ]:
candidate_observed = []
candidate = {item["id"]: request(item["text"], demo_response, candidate_observed, include_vacation=True) for item in fixtures}
for item in fixtures:
    name = item["id"]
    pprint({"id": name, "draft_expected": expectations["expected_decisions"][name],
            "baseline": baseline[name]["decision"], "candidate": candidate[name]["decision"], "reason": candidate[name]["reason"]})

## Count draft disagreements, then explain the trade-off

Accidental block means draft allow but actual block; missed violation means draft block but actual allow. These counts describe this tiny synthetic sample. More vocabulary can create new collisions outside it.

In [ ]:
for label, results in [("baseline", baseline), ("candidate", candidate)]:
    accidental_blocks = sum(expectations["expected_decisions"][name] == "allow" and result["decision"] == "block" for name, result in results.items())
    missed_violations = sum(expectations["expected_decisions"][name] == "block" and result["decision"] == "allow" for name, result in results.items())
    pprint({"setting": label, "accidental_blocks_vs_AI_draft": accidental_blocks, "missed_violations_vs_AI_draft": missed_violations})
assert candidate["borderline"]["decision"] == "allow"
assert candidate["leave_collision"]["decision"] == "allow"